<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c37-header.png" alt="Nextia Learning · Recommendation and Personalisation: Show Each Person What Matters" width="100%">

# Tune without fooling yourself

**[Tune without fooling yourself](https://learning.nextia-ai.com/courses/recommendations/m05/tune-without-fooling-yourself/)** · Recommendation and Personalisation: Show Each Person What Matters · Module 5, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** tune ALS on the validation period, test the chosen setting once, and compare it with trending by segment.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (about 8 MB). No account and no key. |
| **You should know** | Training ALS with implicit, from [ALS and BPR with implicit](https://learning.nextia-ai.com/courses/recommendations/m05/als-and-bpr-with-implicit/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, SciPy 1.18.1 and matplotlib 3.11.2 (implicit 0.7.3 or LightGBM 4.7.0 where the lesson needs it), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/recommendations/m05/tune-without-fooling-yourself/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C37/M05-L03-tune-without-fooling-yourself/tune-without-fooling-yourself-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, scipy 1.18.1, implicit 0.7.3, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 scipy==1.18.1 matplotlib==3.11.2 implicit==0.7.3
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas', 'scipy', 'implicit']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 8 MB) into a folder `larkfield-recs` and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-recs` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop, and its shop log is **synthetic**: our code made the products, the customers and every view, basket and order, with a model of customer behaviour. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C37/data/dataset-card.md) explains how.

In [ ]:
# Setup: download the files into larkfield-recs/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C37/"
FILES = {
    "data/products.csv": "3da37a3809f6067da12eb5778b8abcf809545599db6959208c72a790861407ff",
    "data/customers.csv": "282446db7097aec2b30c27c574a6c7b13a8cffce8053fed212d80bf86cc1411c",
    "data/events.csv.gz": "3f46ddfff683586ab1548bc0e6756d85a439a1de941e6de3030edc383d06d1b3",
    "data/splits.json": "c58597803ced6fa787ff23c86d8ee6e9f0c1d7024c0ec109d2cca25caf6213e6",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-recs").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-recs and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-recs.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, SciPy's sparse matrices as `sp` and matplotlib. It defines the course's small helpers, the same ideas as the course project:

- `products()`, `customers()` and `events()` read the data. `events()` has one row per view, basket add or order, with the column `day` (days since 1 February 2025).
- `split(period)` gives the events before the period (`train`, to learn from) and, for each customer who ordered in the period, the products they ordered (`truth`). The periods are `"validation"`, `"test"` and `"launch"`.
- `recall_at_k(list, ordered, k)`, `ndcg_at_k(list, ordered, k)` and `coverage(lists, n_products)` measure suggestion lists.
- `show(ids)` prints products with their title and subcategory.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, the time split, the measures, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
START = pd.Timestamp("2025-02-01")

def products():
    """One row per product (2,994)."""
    df = pd.read_csv("data/products.csv", keep_default_na=False)
    for col in ("age_restricted", "sensitive", "in_stock"):
        df[col] = df[col].astype(str).eq("True")
    return df

def customers():
    """One row per customer (40,000)."""
    return pd.read_csv("data/customers.csv")

def events():
    """One row per event: view, basket or order. day = days since 1 February 2025."""
    ev = pd.read_csv("data/events.csv.gz", parse_dates=["timestamp"])
    ev["day"] = (ev["timestamp"] - START).dt.days
    return ev

def split(period, ev=None):
    """train = the events before the period; truth = {customer: set of products ordered in the period}."""
    ev = events() if ev is None else ev
    first, last = json.load(open("data/splits.json"))[period]
    a, b = (pd.Timestamp(first) - START).days, (pd.Timestamp(last) - START).days
    train = ev[ev["day"] < a]
    target = ev[(ev["day"] >= a) & (ev["day"] <= b) & (ev["event"] == "order")]
    truth = target.groupby("customer_id")["product_id"].agg(set).to_dict()
    return train, truth

def recall_at_k(rec, ordered, k=10):
    """Share of the ordered products that are in the first k suggestions."""
    return len(set(list(rec)[:k]) & set(ordered)) / len(ordered)

def ndcg_at_k(rec, ordered, k=10):
    """DCG of the list divided by the best possible DCG (discount log2(place + 1))."""
    gains = [1.0 if p in ordered else 0.0 for p in list(rec)[:k]]
    dcg = sum(g / np.log2(i + 2) for i, g in enumerate(gains))
    ideal = sum(1 / np.log2(i + 2) for i in range(min(len(ordered), k)))
    return dcg / ideal

def coverage(lists, n_products, k=10):
    """Share of the catalogue that appears in at least one list."""
    return len({p for rec in lists for p in list(rec)[:k]}) / n_products

def rounded(x, n=3):
    """x rounded to n places (None or ... unchanged: the task is not done yet)."""
    return x if x is None or x is ... else round(float(x), n)

def show(ids, n=10):
    """Print products: place, ID, title, subcategory."""
    global _PR
    if "_PR" not in globals():
        _PR = products().set_index("product_id")
    for i, p in enumerate(list(ids)[:n], 1):
        print(f"{i:2}. {p}  {_PR.loc[p, 'title'][:52]:<52}  {_PR.loc[p, 'subcategory']}")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Why the best of many tries looks too good

Every measurement on about 2,000 customers contains some chance. If you try 18 settings and keep the best number, you also keep the luckiest one. Run the next cell: 18 settings that are **all equally good**.

In [ ]:
# A tiny simulation we made: 18 settings that are all equally good (true recall 0.150).
# Each measurement adds chance, like scoring 2,000 customers (standard deviation 0.007).
rng = np.random.default_rng(37)
true_recall = np.full(18, 0.150)
first_try = true_recall + rng.normal(0, 0.007, 18)
best = int(np.argmax(first_try))
print("best of 18 on the first measurement:", round(float(first_try[best]), 3))
second_try = true_recall + rng.normal(0, 0.007, 18)          # new customers, the same settings
print("the same setting, measured again:   ", round(float(second_try[best]), 3))

> **Predict.** If you repeat the experiment 1,000 times, what is the mean of the best of 18? Exactly 0.150, or more?

In [ ]:
wins = []
for _ in range(1000):
    m = ...            # 18 new measurements: true_recall plus rng.normal(0, 0.007, 18)
    best_of_18 = ...   # the best of the 18
    wins.append(best_of_18)
mean_best = float(np.mean(wins)) if all(w is not ... for w in wins) else None
print(mean_best)

In [ ]:
expect('mean_best', rounded(mean_best, 3), 0.163, tolerance=0.002)

> **Check.** The best of 18 is about 0.163 on average, although every setting is 0.150. The winner's number is too high. Measure it again on new data, and it falls back. That is why we choose on one period and report on another.

## 2. Two periods, built as the project builds them

The next cell defines `period(name)`: the matrix of the events before the period, its customers and their orders, the products on sale, the trending fallback and the segments. `als_recalls()` trains ALS and returns each customer's recall@10. Run it (about 10 seconds).

In [ ]:
import time
from implicit.als import AlternatingLeastSquares
ev = events()
pr, cu = products(), customers()
p_index = pd.Series(np.arange(len(pr)), index=pr["product_id"])
c_index = pd.Series(np.arange(len(cu)), index=cu["customer_id"])
ids = pr["product_id"].to_numpy()

def period(name):
    """Everything the project uses for one period: the matrix, the customers, their orders, the fallback."""
    train, truth = split(name, ev)
    w = train["event"].map({"view": 1.0, "basket": 3.0, "order": 5.0}).to_numpy()
    M = sp.csr_matrix((w, (c_index[train["customer_id"]].to_numpy(), p_index[train["product_id"]].to_numpy())),
                      shape=(len(cu), len(pr)), dtype=np.float32)
    M.sum_duplicates()
    first_day = (pd.Timestamp(json.load(open("data/splits.json"))[name][0]) - START).days
    on_sale = ((pd.to_datetime(pr["added"]) - START).dt.days <= first_day).to_numpy()
    recent = train[(train["event"] == "order") & (train["day"] >= first_day - 28)]
    count = recent.drop_duplicates(["customer_id", "product_id"]).groupby("product_id").size()
    count = count.reindex(pr["product_id"], fill_value=0).to_numpy().astype(float)
    count[~on_sale] = 0
    trending = [i for i in np.argsort(-count, kind="stable") if count[i] > 0][:10]
    users = sorted(truth)
    before = train[train["event"] == "order"].groupby("customer_id")["session_id"].nunique()
    n = before.reindex(users, fill_value=0).to_numpy()
    return {"M": M, "truth": truth, "users": users, "rows": c_index[users].to_numpy(), "on_sale": on_sale,
            "trending": trending, "known": np.asarray(M.sum(axis=1)).ravel() > 0,
            "segment": np.where(n == 0, "new", np.where(n <= 2, "occasional", "loyal"))}

def als_recalls(P, factors, regularization, alpha):
    """Train ALS on the period's matrix and return each customer's recall@10."""
    model = AlternatingLeastSquares(factors=factors, regularization=regularization, alpha=alpha, iterations=15,
                                    random_state=37, num_threads=1)
    model.fit(P["M"], show_progress=False)
    s = np.asarray(model.user_factors)[P["rows"]] @ np.asarray(model.item_factors).T
    s[:, ~P["on_sale"]] = -np.inf
    top = np.argsort(-s, axis=1, kind="stable")[:, :10]
    lists = [ids[t] if P["known"][r] else ids[P["trending"]] for t, r in zip(top, P["rows"])]
    return np.array([recall_at_k(l, P["truth"][u]) for l, u in zip(lists, P["users"])])

val = period("validation")
print(f"validation: {len(val['users']):,} customers")

## 3. Tune on the validation period only

Four settings: 32 or 64 numbers per vector, alpha 1 or 2, regularisation 0.1. Each one learns from the events before 8 June and is scored on 8 to 21 June. The test period is not touched. Run the cell (about 15 seconds).

In [ ]:
results = {}
for factors in (32, 64):
    for alpha in (1.0, 2.0):
        t0 = time.perf_counter()
        r = als_recalls(val, factors, 0.1, alpha)
        results[(factors, alpha)] = float(r.mean())
        print(f"factors {factors:3}  alpha {alpha}  validation recall@10 {r.mean():.3f}  ({time.perf_counter() - t0:.1f} s)")

## 4. Your turn: test once

Choose the setting with the best validation recall. Then train it again on every event before the test period and score the test period **once**.

> **Predict.** Will the test recall be higher, the same, or lower than the best validation recall?

In [ ]:
best = ...                         # the key of `results` with the highest validation recall
best_factors, best_alpha = best if best is not ... else (None, None)
test = period("test")              # learn from every event before 22 June, score the test period
test_recalls = ...                 # als_recalls(test, ...) with the chosen setting and regularization 0.1
test_recall = float(test_recalls.mean()) if test_recalls is not ... else None
print(best_factors, best_alpha, test_recall)

In [ ]:
expect('test_recall', rounded(test_recall, 3), 0.149, tolerance=0.002)

> **Check.** The best setting is factors 32, alpha 1.0, with validation recall 0.169. On the test period, it has 0.149: lower. Part of the fall is the luck of section 1, and part is that the two periods are different weeks.

## 5. Compare with the baseline, by segment

Run the next cell. It compares the tuned ALS with trending on the same test customers.

In [ ]:
if test_recalls is ...:
    print("Do the task of section 4 first.")
else:
    t = test["trending"]
    trend = np.array([recall_at_k(ids[t], test["truth"][u]) for u in test["users"]])
    table = pd.DataFrame({"segment": test["segment"], "tuned ALS": test_recalls, "trending": trend})
    print(table.groupby("segment").mean().reindex(["new", "occasional", "loyal"]).round(3))
    print("all customers:", table[["tuned ALS", "trending"]].mean().round(3).to_dict())

> **Check.** Tuned ALS (0.149) is still below trending (0.156) on all customers. It is a little better for loyal customers and worse for new and occasional ones. The project's `recs compare` gives each difference an interval: none of them is clear.

In the project: `recs tune als --grid ...` and then `recs evaluate als --set ...` once, as the lesson shows.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Content-based recommendations](https://learning.nextia-ai.com/courses/recommendations/m06/content-based-recommendations/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/recommendations/m05/tune-without-fooling-yourself/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The Larkfield shop log is synthetic: our code made the products, the customers and everything they did, with a model of customer behaviour. No real customer is in this data.